# Capstone · Phase 6：系统实现与论文撰写 -- 营销智能体系统的完整交付

**版本**：v5.0 学习材料包（Capstone收官Phase）
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **DSR六步框架** 将Phase 1-5整合为完整Capstone artifact，产出可复现的研究贡献
2. 用 **LangSmith @traceable** 追踪系统执行链，构建可复现研究的trace存档基础设施
3. 用 **statsmodels + arxiv** 撰写IMRaD论文的Results统计报告与文献对比
4. 用 **deepeval LLM-as-a-judge** 评估论文草稿质量，制定学术发表路线图
5. 理解**天道推演×多Agent仿真**作为Capstone特色理论视角的同构关系

## 说明
本笔记本有 **7 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：langsmith（可复现trace）+ deepeval（LLM-as-a-judge）+ statsmodels（统计报告）+ arxiv（文献对比）+ causaldata/dowhy（Phase 1-5数据整合）。
Capstone场景：AI营销Agent系统的完整实现 + IMRaD论文 + 发表路线图。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> 所有自定义评估（BaseMetric）和统计工具不需要 API Key，可直接运行。
> GEval（LLM-as-a-judge）需要 OPENAI_API_KEY，本笔记本提供 fallback 模式。

In [ ]:
# !pip install langsmith deepeval statsmodels arxiv causaldata dowhy -q

import warnings
warnings.filterwarnings('ignore')
import os
os.environ["LANGSMITH_TRACING"] = "false"  # local mode, no API key needed

import pandas as pd
import numpy as np
from typing import TypedDict, Dict, Any

# 因果推断（Phase 4整合）
from causaldata import nsw_mixtape
from dowhy import CausalModel

# 可复现研究基础设施（Phase 6核心）
from langsmith import traceable

# 论文Results统计报告
import statsmodels.api as sm
from scipy import stats

# arxiv文献对比
import arxiv

# LLM-as-a-judge论文评估
from deepeval.metrics import BaseMetric, GEval
from deepeval.test_case import LLMTestCase
try:
    from deepeval.test_case import LLMTestCaseParams
except ImportError:
    from deepeval.test_case import SingleTurnParams as LLMTestCaseParams

print("导入完成")
print("Capstone Phase 6: langsmith + deepeval + statsmodels + arxiv")
print("整合Phase 1-5: causaldata + DoWhy")

## 1. DSR Artifact 设计：Phase 1-5整合

本Phase是Capstone的最终交付。用 **DSR六步框架**（Hevner et al. 2004; Peffers et al. 2007）将Phase 1-5整合为一个完整的设计科学artifact。

### Phase 1-5映射
| Phase | 能力 | 在Capstone中的角色 | 真实库/方法 |
|:-----:|------|-------------------|------------|
| Phase 1 | 问题定义+文献综述 | 研究问题+PRISMA综述 | arxiv文献检索 |
| Phase 2 | 数据表示+知识图谱 | 客户/产品向量化表示 | embeddings |
| Phase 3 | Agentic系统架构 | LangGraph Agent编排 | LangGraph |
| Phase 4 | 因果实验设计 | ATE估计+反事实 | DoWhy + causaldata |
| Phase 5 | 商业模式+价值评估 | ROI+价值捕获 | 商业模式分析 |
| **Phase 6** | **系统实现+论文撰写** | **整合交付+IMRaD+发表** | **langsmith+deepeval+statsmodels** |

### DSR六步框架
```
Step 1: 问题识别 -> Step 2: 目标定义 -> Step 3: 设计开发
Step 4: 演示     -> Step 5: 评估    -> Step 6: 传播
```
你的Capstone就是一个 **DSR artifact** -- 它的架构模式、评估方法、部署经验都是可发表的知识贡献。

In [ ]:
# TODO 1：DSR Artifact 设计 -- 用DSR六步框架定义Capstone
# 提示：定义dsr_plan字典，六步各写1-2句描述，整合Phase 1-5
#   参考：Hevner et al. (2004) MIS Quarterly; Peffers et al. (2007) JMIS
#   教材：../../../AI原生化商业博士_独立教材_Capstone_AI和商业分析项目.md § Phase 6
# 要求：为"AI营销Agent系统的完整实现+IMRaD论文"Capstone定义DSR计划

# ===== 你的代码 =====
dsr_plan = {
    "problem_identification": "",   # 营销面临什么问题？
    "objectives": "",               # artifact应达到什么效果？
    "design_development": "",       # 架构/Agent/评估设计
    "demonstration": "",            # 在什么场景演示？
    "evaluation": "",               # 评估方法
    "communication": ""             # 如何传播
}
raise NotImplementedError("完成后删除此行，填入你的代码")
# ====================

print("DSR六步计划：")
for step, desc in dsr_plan.items():
    print(f"  [{step}] {desc[:60]}...")

## 2. LangSmith 可复现研究基础设施

**核心问题**：如何让他人能独立复现你的Agent系统行为？

**答案**：用 LangSmith 的 `@traceable` 装饰器追踪系统执行的完整调用链，生成trace存档。

```
@traceable pipeline:
  Phase 1: load_research_question()  -> 研究问题
  Phase 2: load_data_representation() -> NSW数据营销映射
  Phase 3: build_agent()             -> Agent架构
  Phase 4: estimate_causal_effect()  -> ATE
  Phase 5: evaluate_business_value() -> ROI
  Phase 6: generate_paper()          -> IMRaD草稿
```

每个步骤都被trace记录，构成**可复现研究的trace存档**--这是2026年前沿的可复现研究实践。

In [ ]:
# TODO 2：LangSmith 可复现研究基础设施 -- 用@traceable追踪系统执行链
# 提示：
#   1. 用@traceable(name="capstone_pipeline")装饰pipeline函数
#   2. 函数内整合Phase 1-5：研究问题->数据->Agent->因果->价值
#   3. 运行pipeline，打印trace结构
# 要求：构建可追踪的Capstone pipeline，产出trace存档

# ===== 你的代码 =====
@traceable(name="capstone_pipeline")
def run_capstone_pipeline(data_desc: str, treatment: str, outcome: str) -> dict:
    """Phase 1-5整合pipeline，被LangSmith trace追踪"""
    pass

pipeline_result = None  # 运行pipeline
raise NotImplementedError("完成后删除此行，填入你的代码")
# ====================

print("Pipeline trace结构：")
for k, v in pipeline_result.items():
    print(f"  {k}: {str(v)[:80]}")

## 3. Phase 1-5 数据整合：NSW真实RCT + DoWhy因果估计

**NSW职业培训实验**：因果推断领域的经典真实RCT数据集（N=445）。

**营销映射**：
| NSW变量 | 营销映射 | 角色 |
|---------|---------|------|
| `treat` | 是否收到个性化营销 | 处理 T |
| `re78` | 营销后转化率/GMV | 结果 Y |
| `re75` | 实验前历史消费 | CUPED协变量 |
| `age`,`educ`,... | 用户画像 | 协变量 X |

这一步整合Phase 2（数据表示）+ Phase 4（因果实验）的产出，作为论文Methods和Results的数据基础。

In [ ]:
# TODO 3：Phase 1-5 数据整合 -- 加载NSW真实RCT + DoWhy因果估计
# 提示：
#   from causaldata import nsw_mixtape
#   df = nsw_mixtape.load_pandas().data
#   model = CausalModel(data=df, treatment=TREATMENT, outcome=OUTCOME, common_causes=COVARIATES)
#   estimand = model.identify_effect(proceed_when_unidentifiable=True)
#   estimate = model.estimate_effect(estimand, method_name="backdoor.linear_regression")
# 要求：加载数据，定义变量，估计ATE，整合为phase_outputs字典

# ===== 你的代码 =====
df = None               # 加载NSW数据
TREATMENT = None        # 处理变量名
OUTCOME = None          # 结果变量名
COVARIATES = None       # 协变量列表
model = None            # 构建CausalModel
estimand = None         # 识别因果效应
estimate = None         # 估计ATE
ate_value = None        # ATE数值
raise NotImplementedError("完成后删除此行，填入你的代码")
# ====================

print(f"数据形状: {df.shape}")
print(f"ATE: {ate_value:.2f}")
print(f"解释: 营销干预使结果{'增加' if ate_value > 0 else '减少'} {abs(ate_value):.2f}")

## 4. 论文 Results 统计报告：statsmodels

用 **statsmodels + scipy** 跑统计检验，生成APA格式的Results部分：
- **独立样本t检验**：营销组 vs 对照组的转化率差异
- **Cohen's d 效应量**：差异的大小（0.2小/0.5中/0.8大）
- **卡方检验**：处理分配是否随机

APA格式示例：`t(df) = X.XX, p < .001, d = X.XX`

In [ ]:
# TODO 4：论文 Results 统计报告 -- 用statsmodels/scipy跑统计检验
# 提示：
#   stats.ttest_ind(treat_grp, ctrl_grp)  # 独立样本t检验
#   Cohen's d = (mean1 - mean2) / pooled_std
#   APA格式: t(df) = X.XX, p < .001, d = X.XX
# 要求：跑t检验+Cohen's d+卡方检验，生成APA格式统计报告

# ===== 你的代码 =====
treat_grp = None   # 处理组re78
ctrl_grp = None    # 对照组re78
t_stat = None      # t统计量
p_value = None     # p值
cohens_d = None    # Cohen's d效应量
apa_result = None  # APA格式字符串
raise NotImplementedError("完成后删除此行，填入你的代码")
# ====================

print(f"处理组均值: {treat_grp.mean():.2f}, 对照组均值: {ctrl_grp.mean():.2f}")
print(f"APA报告: {apa_result}")
effect_size = "小" if abs(cohens_d) < 0.5 else ("中" if abs(cohens_d) < 0.8 else "大")
print(f"效应量解读: d={cohens_d:.3f} ({effect_size})")

## 5. IMRaD 论文草稿生成

把前面所有产出整合为IMRaD结构论文草稿：

| 章节 | 内容 | DSR映射 | 数据来源 |
|------|------|---------|---------|
| Introduction | 研究问题+贡献 | DSR Step 1 | Phase 1 |
| Methods | 系统架构+评估方法 | DSR Step 3 | Phase 2-3 |
| Results | ATE+统计检验+Agent评估 | DSR Step 5 | Phase 4-5 |
| Discussion | 发现+局限+未来+天道推演 | DSR Step 6 | Phase 6 |

论文标题模板：`[方法] for [问题]: A [框架] Approach`

In [ ]:
# TODO 5：IMRaD 论文草稿 -- 整合所有产出生成结构化论文
# 提示：
#   1. 定义generate_paper_draft函数，整合dsr_plan/ate/stats/eval
#   2. 用IMRaD结构：Introduction/Methods/Results/Discussion
#   3. Methods中包含DSR artifact描述
#   4. Discussion中包含天道推演×多Agent仿真特色章节
# 要求：生成结构化论文草稿（每节3-5句）

# ===== 你的代码 =====
def generate_paper_draft(dsr_plan, ate_value, apa_result, cohens_d, n_samples):
    """生成IMRaD论文草稿"""
    draft = None
    return draft

paper_draft = None  # 调用函数生成论文草稿
raise NotImplementedError("完成后删除此行，填入你的代码")
# ====================

print(f"论文标题: {paper_draft['title']}")
print(f"摘要字数: {len(paper_draft['abstract'])}")
for section in ["introduction", "methods", "results", "discussion"]:
    print(f"\n[{section.upper()}] {paper_draft[section][:100]}...")

## 6. 论文质量评估：deepeval LLM-as-a-judge

用 **deepeval** 评估生成的IMRaD论文草稿质量：

| 评估维度 | 说明 | 为什么重要 |
|---------|------|------------|
| IMRaD完整性 | 四部分是否齐全 | 论文结构要求 |
| 有统计依据 | 是否引用ATE/p值 | Results数据说话 |
| 有DSR描述 | 是否含artifact设计 | DSR贡献声明 |
| 有可复现性 | 是否提及trace/代码 | 可复现研究要求 |
| 有发表路线 | 是否含投稿计划 | 传播策略 |

用自定义 **BaseMetric**（不需要API Key）做规则检查，
进阶可用 **GEval**（LLM-as-a-judge，需OPENAI_API_KEY）做语义评估。

In [ ]:
# TODO 6：论文质量评估 -- 用deepeval LLM-as-a-judge评估论文草稿
# 提示：
#   1. 创建LLMTestCase包装论文草稿
#   2. 自定义BaseMetric评估IMRaD完整性/统计依据/DSR描述/可复现性
#   3. （可选）用GEval做LLM-as-a-judge语义评估（需OPENAI_API_KEY）
# 要求：定义PaperQualityMetric，评估论文质量，打印评分

# ===== 你的代码 =====
class PaperQualityMetric(BaseMetric):
    """评估IMRaD论文质量：结构完整性/统计依据/DSR描述/可复现性/发表路线"""
    def __init__(self, threshold=0.7):
        self.threshold = threshold
    pass

test_case = None       # 用LLMTestCase包装论文草稿
metric = None          # 创建PaperQualityMetric
eval_score = None      # 评估分数
raise NotImplementedError("完成后删除此行，填入你的代码")
# ====================

print(f"论文质量评分: {eval_score:.2f}")
print(f"评估通过: {metric.is_successful()}")
print(f"评估理由: {metric.reason}")

## 7. arxiv 文献对比 + 发表路线图 + 天道推演特色章节

### arxiv 文献对比
用 **arxiv** Python包搜索相关论文，与你的Capstone做对比定位。

### 学术发表路线图
```
Step 1: arXiv预印本（立即）-> 建立优先权
Step 2: 投稿会议（3-6月）-> ICIS / HICSS
Step 3: 投稿期刊（会议后）-> Decision Support Systems
Step 4: 持续迭代（2-3轮审稿）
```

### 天道推演×多Agent仿真（特色章节）

> 本节与项目CLAUDE.md的「天道推演系统」同构，作为Capstone的特色理论视角。

天道推演的沙盘模拟（因果链追踪+多路径概率评估）与多Agent仿真（Agent交互+涌现行为预测）共享同一因果建模底层。你的营销Agent系统本质上是一个计算化的天道推演沙盘。

In [ ]:
# TODO 7：arxiv 文献对比 + 发表路线图 + 天道推演特色章节
# 提示：
#   1. 用arxiv.Search搜索"causal inference marketing agent"相关论文
#   2. 生成发表路线图（arXiv->会议->期刊）
#   3. 撰写天道推演×多Agent仿真特色章节
# 要求：搜索arxiv文献，生成发表路线图和特色章节

# ===== 你的代码 =====
arxiv_papers = None    # 搜索arxiv相关论文
publication_roadmap = None  # 发表路线图
tiandao_chapter = None # 天道推演×多Agent仿真特色章节
raise NotImplementedError("完成后删除此行，填入你的代码")
# ====================

print(f"arxiv相关论文: {len(arxiv_papers)}篇")
for p in arxiv_papers[:3]:
    print(f"  - {p['title'][:70]}")
print(f"\n发表路线图: {len(publication_roadmap)}步")
print(f"\n天道推演章节: {tiandao_chapter[:100]}...")

## 8. 反思与前沿

### 反思问题
1. 你的Capstone在DSR六步中哪一步最薄弱？如何改进？
2. LangSmith的trace存档如何提升可复现性？有哪些局限？
3. LLM-as-a-judge评估论文质量，与真实同行评审的差距在哪？
4. 天道推演×多Agent仿真的同构关系，如何用形式化语言描述？

### 2026前沿
- **DSR artifact**：Agent系统作为可发表的设计科学贡献
- **可复现研究**：trace存档 + 开源代码 + 测试套件
- **天道推演×多Agent仿真**：因果链追踪 + 涌现行为预测的同构
- **LLM-as-a-judge**：用LLM自动评估论文质量（NeurIPS 2023, arXiv 2306.05685）

参考 [Hevner 2004](https://www.jstor.org/stable/25148625) + [Peffers 2007](https://desrist.org/desrist/files/peffers2007.pdf) + [LangSmith](https://docs.smith.langchain.com/) + [deepeval](https://github.com/confident-ai/deepeval)